In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\HP\nhs_a&e_performance analysis\Data\Processed\nhs_ae_analysis.db")

In [5]:
##Preview table data

pd.read_sql_query("""
SELECT *
FROM monthly_ae
LIMIT 10;
""", conn)

,month,org_code,org_name,total_attendances,over_4hrs
0,2025-04-01 00:00:00,NDJ,FIRST COMMUNITY HEALTH AND CARE CIC,0,0
1,2025-04-01 00:00:00,NTV0B,ASHFORD WALK-IN-CENTRE,0,0
2,2025-04-01 00:00:00,NTV0W,WOKING WALK IN CENTRE,0,0
3,2025-04-01 00:00:00,Y02615,SOUTH BIRMINGHAM GP WALK IN CENTRE,0,0
4,2025-04-01 00:00:00,AQN04,PHL LYMINGTON UTC,0,0
5,2025-04-01 00:00:00,C82038,LATHAM HOUSE MEDICAL PRACTICE,0,0
6,2025-04-01 00:00:00,NQTE4,SUMMERFIELD URGENT CARE CENTRE,0,0
7,2025-04-01 00:00:00,RBQ,LIVERPOOL HEART AND CHEST HOSPITAL NHS FOUNDAT...,0,0
8,2025-04-01 00:00:00,RCD,HARROGATE AND DISTRICT NHS FOUNDATION TRUST,4918,1081
9,2025-04-01 00:00:00,RJ8,CORNWALL PARTNERSHIP NHS FOUNDATION TRUST,0,0


In [6]:
## Check the date range

pd.read_sql_query("""
SELECT
MIN(month) AS first_month,
MAX(month) AS last_month,
COUNT(DISTINCT month) AS number_of_months,
COUNT(DISTINCT org_code) AS number_of_providers
FROM monthly_ae;
""", conn)

,first_month,last_month,number_of_months,number_of_providers
0,2025-04-01 00:00:00,2026-03-01 00:00:00,12,206


In [7]:
## Monthly record counts

pd.read_sql_query("""
SELECT
month,
COUNT(*) AS records,
COUNT(DISTINCT org_code) AS providers,
SUM(total_attendances) AS total_attendances
FROM monthly_ae
GROUP BY month
ORDER BY month;
""", conn)

,month,records,providers,total_attendances
0,2025-04-01 00:00:00,199,199,2709918
1,2025-05-01 00:00:00,201,201,2838612
2,2025-06-01 00:00:00,202,202,2790660
3,2025-07-01 00:00:00,202,202,2866360
4,2025-08-01 00:00:00,202,202,2703262
5,2025-09-01 00:00:00,202,202,2770858
6,2025-10-01 00:00:00,198,198,2898960
7,2025-11-01 00:00:00,198,198,2840066
8,2025-12-01 00:00:00,198,198,2815282
9,2026-01-01 00:00:00,198,198,2810070


In [8]:
## Check for missing values

pd.read_sql_query("""
SELECT *
FROM monthly_ae
WHERE month IS NULL
   OR org_code IS NULL
   OR org_name IS NULL
   OR total_attendances IS NULL
   OR over_4hrs IS NULL;
""", conn)

,month,org_code,org_name,total_attendances,over_4hrs


In [9]:
## Check for excact duplicates

pd.read_sql_query("""
SELECT month, org_code, org_name, total_attendances, over_4hrs,
COUNT(*) AS duplicate_count
FROM monthly_ae
GROUP BY month, org_code, org_name, total_attendances, over_4hrs
HAVING COUNT(*) > 1;
""", conn)

,month,org_code,org_name,total_attendances,over_4hrs,duplicate_count


In [10]:
##Check for negative values

pd.read_sql_query("""
SELECT *
FROM monthly_ae
WHERE total_attendances < 0
   OR over_4hrs < 0;
""", conn)

,month,org_code,org_name,total_attendances,over_4hrs


In [12]:
## Check for invalid four-hour figures

pd.read_sql_query("""
SELECT * FROM monthly_ae
WHERE over_4hrs > total_attendances;
""", conn)

,month,org_code,org_name,total_attendances,over_4hrs


In [13]:
### Data analysis
## Monthly A&E performance

monthly_summary = pd.read_sql_query("""
SELECT month,
SUM(total_attendances) AS total_attendances, SUM(over_4hrs) AS over_4hrs, SUM(total_attendances - over_4hrs) 
AS within_4hrs,
ROUND(100.0 * SUM(total_attendances - over_4hrs) / NULLIF(SUM(total_attendances), 0), 2) AS within_4hrs_pct

FROM monthly_ae
WHERE org_code = 'TOTAL'
GROUP BY month
ORDER BY month;
""", conn)

display(monthly_summary)

,month,total_attendances,over_4hrs,within_4hrs,within_4hrs_pct
0,2025-04-01 00:00:00,1354959,544820,810139,59.79
1,2025-05-01 00:00:00,1419306,555982,863324,60.83
2,2025-06-01 00:00:00,1395330,539355,855975,61.35
3,2025-07-01 00:00:00,1433180,534607,898573,62.70
4,2025-08-01 00:00:00,1351631,518160,833471,61.66
5,2025-09-01 00:00:00,1385429,543443,841986,60.77
6,2025-10-01 00:00:00,1449480,581850,867630,59.86
7,2025-11-01 00:00:00,1420033,566630,853403,60.10
8,2025-12-01 00:00:00,1407641,572315,835326,59.34
9,2026-01-01 00:00:00,1405035,603633,801402,57.04


In [14]:
## Overall national performance

annual_summary = pd.read_sql_query("""
SELECT
SUM(total_attendances) AS total_attendances, SUM(over_4hrs) AS over_4hrs, SUM(total_attendances - over_4hrs) AS within_4hrs,
ROUND(100.0 * SUM(total_attendances - over_4hrs)/ NULLIF(SUM(total_attendances), 0),2) 
AS within_4hrs_pct

FROM monthly_ae
WHERE org_code = 'TOTAL';
""", conn)

display(annual_summary)

,total_attendances,over_4hrs,within_4hrs,within_4hrs_pct
0,16744864,6604765,10140099,60.56


In [15]:
## Identify the best and worst performing months

pd.read_sql_query("""
SELECT month,
ROUND(100.0 * (total_attendances - over_4hrs)/ NULLIF(total_attendances, 0), 2) 
AS within_4hrs_pct

FROM monthly_ae
WHERE org_code = 'TOTAL'
ORDER BY within_4hrs_pct DESC;
""", conn)

,month,within_4hrs_pct
0,2026-03-01 00:00:00,63.89
1,2025-07-01 00:00:00,62.70
2,2025-08-01 00:00:00,61.66
3,2025-06-01 00:00:00,61.35
4,2025-05-01 00:00:00,60.83
5,2025-09-01 00:00:00,60.77
6,2025-11-01 00:00:00,60.10
7,2025-10-01 00:00:00,59.86
8,2025-04-01 00:00:00,59.79
9,2025-12-01 00:00:00,59.34


In [20]:
## Annual performance for providers

provider_summary = pd.read_sql_query("""
SELECT org_code, org_name, 
SUM(total_attendances) AS total_attendances,
SUM(over_4hrs) AS over_4hrs,
SUM(total_attendances - over_4hrs) AS within_4hrs,
ROUND( 100.0 * SUM(total_attendances - over_4hrs) / NULLIF(SUM(total_attendances), 0), 2) 
AS within_4hrs_pct

FROM monthly_ae
WHERE org_code <> 'TOTAL'
GROUP BY org_code, org_name
ORDER BY total_attendances DESC;
""", conn)

display(provider_summary)

,org_code,org_name,total_attendances,over_4hrs,within_4hrs,within_4hrs_pct
0,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,414623,168412,246211,59.38
1,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,343747,118279,225468,65.59
2,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,321536,124651,196885,61.23
3,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,313776,105321,208455,66.43
4,RM3,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,313494,123579,189915,60.58
...,...,...,...,...,...,...
204,Y02615,SOUTH BIRMINGHAM GP WALK IN CENTRE,0,0,0,NaN
205,Y02676,BRIGHTON STATION HEALTH CENTRE,0,0,0,NaN
206,Y03007,ERDINGTON GP HEALTH & WELLBEING WIC,0,0,0,NaN
207,Y03218,THE PINN UNREGISTERED WIC,0,0,0,NaN


In [19]:
## Provider with the highest attendance

pd.read_sql_query("""
SELECT org_code, org_name,
SUM(total_attendances) AS total_attendances

FROM monthly_ae
WHERE org_code <> 'TOTAL'
GROUP BY org_code, org_name
ORDER BY total_attendances DESC
LIMIT 10;
""", conn)

,org_code,org_name,total_attendances
0,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,414623
1,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,343747
2,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,321536
3,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,313776
4,RM3,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,313494
5,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,309660
6,R1H,BARTS HEALTH NHS TRUST,305760
7,RDU,FRIMLEY HEALTH NHS FOUNDATION TRUST,271749
8,RWE,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,264994
9,RQM,CHELSEA AND WESTMINSTER HOSPITAL NHS FOUNDATIO...,237084


In [21]:
## Provider with the lowest four-hour performance

pd.read_sql_query("""
SELECT org_code, org_name,
SUM(total_attendances) AS total_attendances,
ROUND(100.0 * SUM(total_attendances - over_4hrs) / NULLIF(SUM(total_attendances), 0), 2) 
AS within_4hrs_pct

FROM monthly_ae
WHERE org_code <> 'TOTAL'
GROUP BY org_code, org_name
HAVING SUM(total_attendances) > 0
ORDER BY within_4hrs_pct ASC
LIMIT 10;
""", conn)

,org_code,org_name,total_attendances,within_4hrs_pct
0,RBT,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,84933,43.64
1,RXW,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,129655,44.46
2,RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,153351,44.49
3,RWA,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,116768,44.52
4,RK9,UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST,107510,45.31
5,RBL,WIRRAL UNIVERSITY TEACHING HOSPITAL NHS FOUNDA...,94474,46.26
6,RVW,NORTH TEES AND HARTLEPOOL NHS FOUNDATION TRUST,59209,46.71
7,RJN,EAST CHESHIRE NHS TRUST,53202,46.93
8,RFS,CHESTERFIELD ROYAL HOSPITAL NHS FOUNDATION TRUST,76171,47.29
9,RAX,KINGSTON AND RICHMOND NHS FOUNDATION TRUST,83358,47.36


In [23]:
monthly_summary.to_csv(r"C:\Users\HP\nhs_a&e_performance analysis/sql/monthly_ae_summary_sql.csv", index=False)

annual_summary.to_csv(r"C:\Users\HP\nhs_a&e_performance analysis/sql/annual_ae_summary_sql.csv", index=False)

provider_summary.to_csv(r"C:\Users\HP\nhs_a&e_performance analysis/sql/provider_annual_summary_sql.csv", index=False)